## Setup

In [1]:
from pathlib import Path
import sys, os, json
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(root_dir))

from src.dataloaders import create_dataloaders
from src.metrics import calculate_macro_f1
from src.labels import load_label_mapping
from src.device import get_default_device

device = get_default_device()
print("device:", device)

models_dir = root_dir / "outputs" / "models" / "densenet121"
metrics_dir = root_dir / "reports" / "metrics" / "densenet121"
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

run_id = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
print(f"run_id: {run_id}")

device: cuda
run_id: 2026-05-12_20-44-41


## DataLoader

Если запускался `just prepare-data-with-heuristics`, эвристики уже включены в processed CSV - `create_dataloaders` подхватит их автоматически через `image_path`.

In [2]:
batch_size = 32
image_size = 224

train_loader, val_loader = create_dataloaders(
    batch_size=batch_size,
    num_workers=2,
    image_size=image_size,
    use_weighted_sampling=True,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 145, Val batches: 15


## Модель DenseNet121

In [3]:
from torchvision.models import densenet121, DenseNet121_Weights

num_classes = 19

model = densenet121(weights=DenseNet121_Weights.DEFAULT)
model.classifier = nn.Linear(model.classifier.in_features, num_classes)
model = model.to(device)
print(f"Classifier: {model.classifier}")

Classifier: Linear(in_features=1024, out_features=19, bias=True)


## Train / Validate functions

In [4]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for batch_idx, (images, targets) in enumerate(loader):
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())
        if batch_idx % 50 == 0:
            print(f"batch {batch_idx}/{len(loader)}  loss={loss.item():.4f}")

    epoch_loss = running_loss / len(loader)
    epoch_f1 = calculate_macro_f1(all_targets, all_preds)
    return epoch_loss, epoch_f1


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for images, targets in loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            running_loss += loss.item()
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.detach().cpu().numpy())
            all_targets.extend(targets.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_f1 = calculate_macro_f1(all_targets, all_preds)
    return epoch_loss, epoch_f1, all_targets, all_preds

## Class weights

In [5]:
from sklearn.utils.class_weight import compute_class_weight

learning_rate = 1e-4
weight_decay  = 1e-4

targets_array = train_loader.dataset.df["result"].values
classes = np.unique(targets_array)

class_weight_values = compute_class_weight("balanced", classes=classes, y=targets_array)
class_weights = np.ones(num_classes, dtype=np.float64)
for cls, w in zip(classes, class_weight_values):
    class_weights[cls] = w

# Boost underrepresented / hard classes
class_weights[2] *= 3.0   # универсальная комната (низкий recall)
class_weights[4] *= 0.7   # спальня (слишком жадная)
class_weights[5] *= 1.5   # кабинет
class_weights[11] *= 1.5   # гардеробная

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(class_weights, dtype=torch.float).to(device),
    label_smoothing=0.1,
)
print("Веса классов сконфигурированы")

Веса классов сконфигурированы


## Этап 1 - head-only (2 эпохи)

In [6]:
head_checkpoint = models_dir / "densenet121_head_best.pth"

for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

optimizer_head = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
)

best_val_f1_head = 0.0
num_epochs_head = 2

print("Этап 1: head-only")
for epoch in range(num_epochs_head):
    print(f"Epoch {epoch + 1}/{num_epochs_head}")
    train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer_head, device)
    val_loss, val_f1, _, _ = validate_one_epoch(model, val_loader, criterion, device)
    print(f"  train_loss={train_loss:.4f}  train_f1={train_f1:.4f} | val_loss={val_loss:.4f}  val_f1={val_f1:.4f}")
    if val_f1 > best_val_f1_head:
        best_val_f1_head = val_f1
        torch.save(model.state_dict(), head_checkpoint)
        print(f"Сохранено (val_f1={val_f1:.4f})")

print(f"Этап 1 завершен. Лучший val_f1={best_val_f1_head:.4f}")

Этап 1: head-only
Epoch 1/2
batch 0/145  loss=3.0589
batch 50/145  loss=2.3646
batch 100/145  loss=2.5081
  train_loss=2.4032  train_f1=0.2624 | val_loss=2.1071  val_f1=0.3756
Сохранено (val_f1=0.3756)
Epoch 2/2
batch 0/145  loss=2.0563
batch 50/145  loss=2.0124
batch 100/145  loss=2.0669
  train_loss=1.9157  train_f1=0.4430 | val_loss=1.9036  val_f1=0.4363
Сохранено (val_f1=0.4363)
Этап 1 завершен. Лучший val_f1=0.4363


## Этап 2 - fine-tuning (8 эпох)

In [7]:
finetune_checkpoint = models_dir / f"densenet121_{run_id}_best.pth"

state_dict = torch.load(head_checkpoint, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

for param in model.parameters():
    param.requires_grad = True

optimizer_finetune = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

best_val_f1_finetune = 0.0
best_val_loss_finetune = None
best_train_loss_finetune = None
best_epoch_finetune = 0
num_epochs_finetune = 8
history = []

print("Этап 2: full fine-tuning")
for epoch in range(num_epochs_finetune):
    print(f"Epoch {epoch + 1}/{num_epochs_finetune}")
    train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer_finetune, device)
    val_loss, val_f1, val_targets, val_preds = validate_one_epoch(model, val_loader, criterion, device)
    print(f"  train_loss={train_loss:.4f}  train_f1={train_f1:.4f} | val_loss={val_loss:.4f}  val_f1={val_f1:.4f}")
    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss, "train_f1": train_f1,
        "val_loss": val_loss, "val_f1": val_f1,
    })
    if val_f1 > best_val_f1_finetune:
        best_val_f1_finetune = val_f1
        best_val_loss_finetune = val_loss
        best_train_loss_finetune = train_loss
        best_epoch_finetune = epoch + 1
        torch.save(model.state_dict(), finetune_checkpoint)
        print(f"Сохранено (val_f1={val_f1:.4f})")

print(f"Этап 2 завершен. Best val_f1={best_val_f1_finetune:.4f}")

Этап 2: full fine-tuning
Epoch 1/8
batch 0/145  loss=1.9737
batch 50/145  loss=1.3541
batch 100/145  loss=1.3429
  train_loss=1.4474  train_f1=0.6443 | val_loss=1.5138  val_f1=0.6047
Сохранено (val_f1=0.6047)
Epoch 2/8
batch 0/145  loss=1.2104
batch 50/145  loss=1.1966
batch 100/145  loss=1.0027
  train_loss=1.1811  train_f1=0.7865 | val_loss=1.4079  val_f1=0.6514
Сохранено (val_f1=0.6514)
Epoch 3/8
batch 0/145  loss=0.9992
batch 50/145  loss=0.8338
batch 100/145  loss=0.9659
  train_loss=0.9852  train_f1=0.8716 | val_loss=1.4306  val_f1=0.6717
Сохранено (val_f1=0.6717)
Epoch 4/8
batch 0/145  loss=1.0333
batch 50/145  loss=0.9448
batch 100/145  loss=0.7803
  train_loss=0.8970  train_f1=0.9124 | val_loss=1.3771  val_f1=0.7062
Сохранено (val_f1=0.7062)
Epoch 5/8
batch 0/145  loss=0.8931
batch 50/145  loss=0.7187
batch 100/145  loss=0.8796
  train_loss=0.8248  train_f1=0.9423 | val_loss=1.4226  val_f1=0.7069
Сохранено (val_f1=0.7069)
Epoch 6/8
batch 0/145  loss=0.9940
batch 50/145  loss=1

## Этап 3 - дообучение с lr=3e-5 (5 эпох)

Продолжаем от лучшего чекпоинта Stage 2 с пониженным lr - плавный дожиг.

In [8]:
state_dict = torch.load(finetune_checkpoint, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

for param in model.parameters():
    param.requires_grad = True

optimizer_stage3 = optim.Adam(model.parameters(), lr=3e-5, weight_decay=weight_decay)

best_val_f1_stage3 = best_val_f1_finetune
best_val_loss_stage3 = best_val_loss_finetune
best_train_loss_stage3 = best_train_loss_finetune
best_epoch_stage3 = best_epoch_finetune
num_epochs_stage3 = 5

print(f"Этап 3: дообучение (старт от val_f1={best_val_f1_stage3:.4f})")
for epoch in range(num_epochs_stage3):
    print(f"Epoch {epoch + 1}/{num_epochs_stage3}")
    train_loss, train_f1 = train_one_epoch(model, train_loader, criterion, optimizer_stage3, device)
    val_loss, val_f1, val_targets, val_preds = validate_one_epoch(model, val_loader, criterion, device)
    print(f"  train_loss={train_loss:.4f}  train_f1={train_f1:.4f} | val_loss={val_loss:.4f}  val_f1={val_f1:.4f}")
    history.append({
        "epoch": f"s3_{epoch + 1}",
        "train_loss": train_loss, "train_f1": train_f1,
        "val_loss": val_loss, "val_f1": val_f1,
    })
    if val_f1 > best_val_f1_stage3:
        best_val_f1_stage3 = val_f1
        best_val_loss_stage3 = val_loss
        best_train_loss_stage3 = train_loss
        best_epoch_stage3 = f"s3_{epoch + 1}"
        torch.save(model.state_dict(), finetune_checkpoint)
        print(f"Сохранено (val_f1={val_f1:.4f})")

print(f"Этап 3 завершен. Best val_f1={best_val_f1_stage3:.4f}")

Этап 3: дообучение (старт от val_f1=0.7127)
Epoch 1/5
batch 0/145  loss=0.7605
batch 50/145  loss=0.5911
batch 100/145  loss=0.6678
  train_loss=0.6782  train_f1=0.9947 | val_loss=1.4001  val_f1=0.7069
Epoch 2/5
batch 0/145  loss=0.6053
batch 50/145  loss=0.6854
batch 100/145  loss=0.7983
  train_loss=0.6603  train_f1=0.9965 | val_loss=1.3814  val_f1=0.7047
Epoch 3/5
batch 0/145  loss=0.5735
batch 50/145  loss=0.6420
batch 100/145  loss=0.7006
  train_loss=0.6638  train_f1=0.9965 | val_loss=1.3806  val_f1=0.7189
Сохранено (val_f1=0.7189)
Epoch 4/5
batch 0/145  loss=0.6340
batch 50/145  loss=0.5995
batch 100/145  loss=0.5995
  train_loss=0.6467  train_f1=0.9971 | val_loss=1.3640  val_f1=0.6953
Epoch 5/5
batch 0/145  loss=0.5999
batch 50/145  loss=0.6300
batch 100/145  loss=0.6549
  train_loss=0.6512  train_f1=0.9983 | val_loss=1.3540  val_f1=0.7245
Сохранено (val_f1=0.7245)
Этап 3 завершен. Best val_f1=0.7245


## Classification report + per-class metrics

In [9]:
from sklearn.metrics import classification_report, accuracy_score
import torch.nn.functional as F

# Загружаем лучшую модель
state_dict = torch.load(finetune_checkpoint, map_location=device, weights_only=True)
model.load_state_dict(state_dict)

# Финальный прогон по val - собираем logits для per-class loss
model.eval()
all_preds, all_targets, all_logits = [], [], []
with torch.no_grad():
    for images, targets in val_loader:
        images, targets = images.to(device), targets.to(device)
        outputs = model(images)
        all_logits.append(outputs.cpu())
        all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

logits_tensor = torch.cat(all_logits, dim=0)
targets_tensor = torch.tensor(all_targets)

label_mapping = load_label_mapping()
unique_classes = sorted(set(all_targets) | set(all_preds))
target_names = [label_mapping.get(i, str(i)) for i in unique_classes]

report_str = classification_report(
    all_targets, all_preds,
    labels=unique_classes,
    target_names=target_names,
    digits=3, zero_division=0,
)
print(report_str)

# Сначала создаём report_dict, потом сохраняем в JSON
report_dict = classification_report(
    all_targets, all_preds,
    labels=unique_classes,
    target_names=target_names,
    output_dict=True,
    zero_division=0,
)

report_json_path = metrics_dir / "densenet121_classification_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report_dict, f, ensure_ascii=False, indent=2)
print(f"JSON отчет сохранен в {report_json_path}")

# Сохраняем текстовый отчет
report_txt_path = metrics_dir / "densenet121_classification_report.txt"
with open(report_txt_path, "w", encoding="utf-8") as f:
    f.write(report_str)
print(f"TXT отчет сохранен в {report_txt_path}")

# Per-class loss - считаем cross-entropy без class_weights и label_smoothing
criterion_per_class = nn.CrossEntropyLoss(reduction="none")
per_sample_loss = criterion_per_class(logits_tensor, targets_tensor).numpy()

per_class_metrics = []
for class_id in unique_classes:
    label = label_mapping.get(class_id, str(class_id))
    mask = np.array(all_targets) == class_id
    class_preds = np.array(all_preds)[mask]
    class_loss = float(per_sample_loss[mask].mean()) if mask.sum() > 0 else 0.0
    class_accuracy = float((class_preds == class_id).mean()) if mask.sum() > 0 else 0.0
    class_f1 = report_dict.get(label, {}).get("f1-score", 0.0)
    per_class_metrics.append({
        "class_id": class_id,
        "label": label,
        "f1": class_f1,
        "accuracy": class_accuracy,
        "loss": class_loss,
        "support": int(mask.sum()),
    })

print("Per-class metrics рассчитаны")

                                      precision    recall  f1-score   support

                    кухня / столовая      0.757     0.824     0.789        34
                      кухня-гостиная      0.462     0.400     0.429        15
               универсальная комната      0.452     0.609     0.519        23
                            гостиная      0.741     0.667     0.702        30
                             спальня      0.760     0.792     0.776        24
                             кабинет      0.737     0.667     0.700        21
                             детская      0.692     0.581     0.632        31
                      ванная комната      0.850     0.739     0.791        23
                              туалет      0.880     0.786     0.830        28
                 совмещенный санузел      0.743     0.867     0.800        30
                  коридор / прихожая      0.786     0.786     0.786        28
гардеробная / кладовая / постирочная      0.586     0.739     0

## Сохранение metrics.json и experiments.json

In [10]:
import numpy as np
from sklearn.metrics import accuracy_score

def convert_to_python_types(obj):
    if isinstance(obj, dict):
        return {k: convert_to_python_types(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_python_types(i) for i in obj]
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

overall_accuracy = accuracy_score(all_targets, all_preds)
final_macro_f1 = calculate_macro_f1(all_targets, all_preds)

# Относительные пути — не зависят от машины и ОС
checkpoint_for_report = finetune_checkpoint.relative_to(root_dir).as_posix()
metrics_path = metrics_dir / "densenet121_metrics.json"
metrics_json_for_report = metrics_path.relative_to(root_dir).as_posix()

metrics_data = {
    "run_id": run_id,
    "model": "densenet121",
    "hyperparameters": {
        "epochs_stage1": num_epochs_head,
        "epochs_stage2": num_epochs_finetune,
        "epochs_stage3": num_epochs_stage3,
        "batch_size": batch_size,
        "image_size": image_size,
        "learning_rate": learning_rate,
        "learning_rate_stage3": 3e-5,
        "weight_decay": weight_decay,
        "label_smoothing": 0.1,
        "pretrained": True,
        "class_weights": True,
        "weighted_sampling": True,
    },
    "best_epoch": best_epoch_stage3,
    "best_macro_f1": best_val_f1_stage3,
    "best_epoch_metrics": {
        "epoch": best_epoch_stage3,
        "train_loss": best_train_loss_stage3,
        "val_loss": best_val_loss_stage3,
        "accuracy": overall_accuracy,
        "macro_f1": final_macro_f1,
        "per_class_metrics": per_class_metrics,
    },
    "checkpoint": checkpoint_for_report,
    "stop_reason": "stage3_complete",
}

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(convert_to_python_types(metrics_data), f, ensure_ascii=False, indent=2)
print(f"Metrics сохранена в {metrics_path}")

experiments_path = metrics_dir / "densenet121_experiments.json"
experiment_entry = {
    "run_id": run_id,
    "model": "densenet121",
    "best_epoch": best_epoch_stage3,
    "best_macro_f1": best_val_f1_stage3,
    "best_accuracy": overall_accuracy,
    "best_train_loss": best_train_loss_stage3,
    "best_val_loss": best_val_loss_stage3,
    "stop_reason": "stage3_complete",
    "checkpoint": checkpoint_for_report,
    "epochs_stage1": num_epochs_head,
    "epochs_stage2": num_epochs_finetune,
    "epochs_stage3": num_epochs_stage3,
    "batch_size": batch_size,
    "image_size": image_size,
    "learning_rate": learning_rate,
    "weight_decay": weight_decay,
    "label_smoothing": 0.1,
    "pretrained": True,
    "class_weights": True,
    "weighted_sampling": True,
    "metrics_json": metrics_json_for_report,
}

if experiments_path.exists():
    with open(experiments_path, "r", encoding="utf-8") as f:
        experiments = json.load(f)
else:
    experiments = []

experiments.append(experiment_entry)
with open(experiments_path, "w", encoding="utf-8") as f:
    json.dump(convert_to_python_types(experiments), f, ensure_ascii=False, indent=2)
print(f"Experiments сохранены в {experiments_path}")
print(f"Финальный отчет:")
print(f"run_id: {run_id}")
print(f"macro_f1: {final_macro_f1:.4f}")
print(f"accuracy: {overall_accuracy:.4f}")
print(f"checkpoint: {checkpoint_for_report}")

Metrics сохранена в C:\Users\pantp\room_type_classifier\reports\metrics\densenet121\densenet121_metrics.json
Experiments сохранены в C:\Users\pantp\room_type_classifier\reports\metrics\densenet121\densenet121_experiments.json
Финальный отчет:
run_id: 2026-05-12_20-44-41
macro_f1: 0.7245
accuracy: 0.7358
checkpoint: outputs/models/densenet121/densenet121_2026-05-12_20-44-41_best.pth
